# Lab 3 — Pandas & Zomato Data Load

**Day 02 · Python for Data Science · Cisco AI/ML Training**

---

## Learning objectives

1. Load a CSV into a **pandas DataFrame** with `read_csv`.
2. Inspect **shape**, **columns**, **dtypes**, and **summary statistics**.
3. Run first-pass **EDA**: `head`, `describe`, `value_counts`.
4. Confirm the **lab profile** row count (**500** restaurants).

> **Checkpoints:** `df.shape == (500, 9)` · mean `aggregate_rating` ≈ **3.70**

**Dataset:** [Zomato restaurants](https://www.kaggle.com/shrutimehta/zomato-restaurants-data) (synthetic **500**-row classroom sample)



## What is a DataFrame?

A **DataFrame** is a 2-D labeled table — think Excel inside Python:

| Concept | Excel | Pandas |
|---------|-------|--------|
| Rows | Records | `index` |
| Columns | Fields | `columns` |
| Types | Number / Text | `dtypes` |
| Filters | AutoFilter | Boolean indexing |

Every ML lab from Day 2 onward starts with: **load → explore → clean → model**.


## Types of ML problems (course topic)

<!-- cisco-topic-coverage -->

| Type | Target | Example in this training |
|------|--------|-------------------------|
| **Regression** | Continuous number | Predict Zomato `aggregate_rating` (Day 2 Lab 5) |
| **Classification** | Category / yes-no | Loan `default` (Days 3–4) |
| **Clustering** | No labels | NYSE symbol segments (Day 5) |
| **Anomaly detection** | Rare events | Credit card `is_fraud` (Day 6) |

Zomato EDA (this lab) supports **supervised** regression and classification later.


---

## 1. Setup paths and imports

We resolve the CSV relative to the `GH` root so the notebook works from `hands-on/day-02/notebooks/`.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

# notebooks/ -> day-02/ -> hands-on/ -> GH/
GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif GH_ROOT.name == "day-02":
    GH_ROOT = GH_ROOT.parents[1]
else:
    # Fallback when opened from repo root or GH/
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "data" / "zomato" / "zomato_restaurants.csv").is_file():
            GH_ROOT = parent
            break

ZOMATO_CSV = GH_ROOT / "data" / "zomato" / "zomato_restaurants.csv"
print("Looking for:", ZOMATO_CSV)
print("Exists:", ZOMATO_CSV.is_file())


---

## 2. Load the CSV

`pd.read_csv` parses delimited text into a DataFrame. Our lab file uses comma separation and a header row.


In [ ]:
df = pd.read_csv(ZOMATO_CSV)

print("Lab 3 — Pandas Zomato load")
print("dataset:", ZOMATO_CSV.name)
print("shape (rows, cols):", df.shape)


### Verify the lab profile

The maintainer toolchain locks classroom data at **500** rows. If you see a different count, you may have the full Kaggle download instead of `GH/data/`.


In [ ]:
EXPECTED_ROWS = 500
EXPECTED_COLS = 9

assert df.shape == (EXPECTED_ROWS, EXPECTED_COLS), (
    f"Expected ({EXPECTED_ROWS}, {EXPECTED_COLS}), got {df.shape}"
)
print("✓ Row/column count matches lab profile")


---

## 3. Column names and data types

Understanding **dtypes** prevents silent bugs (e.g. treating `"Yes"`/`"No"` as numbers).


In [ ]:
print("Columns:", list(df.columns))
print()
print(df.dtypes)


| Column | Role | Dtype |
|--------|------|-------|
| `restaurant_id` | Primary key | object (string) |
| `name` | Label | object |
| `city` | Categorical | object |
| `cuisines` | Categorical | object |
| `aggregate_rating` | **Target** (later labs) | float64 |
| `votes` | Numeric feature | int64 |
| `average_cost_for_two` | Numeric feature | int64 |
| `online_order` | Binary category | object |
| `book_table` | Binary category | object |


---

## 4. First look — `head` and `tail`

`head(n)` shows the first *n* rows; useful for sanity checks after load.


In [ ]:
print("head(3):")
display(df.head(3))

print("\n tail(2):")
display(df.tail(2))


**Observation:** Ratings range roughly 2.5–4.9; cities include Bengaluru, Mumbai, etc. Cuisines are single-label in this synthetic sample.


---

## 5. Missing values

Real Zomato data has nulls; our lab generator produces complete rows — still good practice to check.


In [ ]:
missing = df.isna().sum()
print("Missing per column:")
print(missing)

print("\nAny missing?", missing.any())


---

## 6. Numeric summary — `describe`

`describe()` computes count, mean, std, min, quartiles, max for numeric columns.


In [ ]:
desc = df.describe().round(2)
print("describe (numeric):")
display(desc)


In [ ]:
mean_rating = df["aggregate_rating"].mean()
print(f"mean aggregate_rating: {mean_rating:.2f}")
assert abs(mean_rating - 3.70) < 0.05, "Mean rating drifted — regenerate lab data?"
print("✓ Mean rating checkpoint OK")


---

## 7. Categorical exploration

For non-numeric columns, use `value_counts()` to see frequency distributions.


In [ ]:
print("Top 5 cities by restaurant count:")
print(df["city"].value_counts().head())

print("\nOnline order distribution:")
print(df["online_order"].value_counts())

print("\nTop cuisines:")
print(df["cuisines"].value_counts().head())


---

## 8. Selecting columns — Series vs DataFrame

| Syntax | Returns |
|--------|---------|
| `df["votes"]` | **Series** (1 column) |
| `df[["votes", "aggregate_rating"]]` | **DataFrame** (2+ columns) |

ML feature matrices usually come from a DataFrame subset.


In [ ]:
votes_series = df["votes"]
features_df = df[["votes", "average_cost_for_two", "aggregate_rating"]]

print("Series shape:", votes_series.shape)
print("DataFrame shape:", features_df.shape)
print(features_df.head(3))


---

## 9. Simple filtering (preview)

Boolean masks select rows meeting a condition — foundation for outlier and segment analysis.


In [ ]:
highly_rated = df[df["aggregate_rating"] >= 4.5]
print(f"Restaurants rated >= 4.5: {len(highly_rated)}")

bengaluru = df[df["city"] == "Bengaluru"]
print(f"Bengaluru restaurants: {len(bengaluru)}")
print("Mean rating in Bengaluru:", round(bengaluru["aggregate_rating"].mean(), 2))


---

## 10. Final checkpoint summary


In [ ]:
print("=" * 50)
print("CHECKPOINT SUMMARY")
print("=" * 50)
print(f"shape: {df.shape}")
print(f"columns ({len(df.columns)}): {list(df.columns)}")
print(f"mean aggregate_rating: {df['aggregate_rating'].mean():.2f}")
print(f"mean votes: {df['votes'].mean():.2f}")
print(f"mean average_cost_for_two: {df['average_cost_for_two'].mean():.2f}")

required = {"aggregate_rating", "votes", "average_cost_for_two", "city", "cuisines"}
assert required.issubset(df.columns)
print("\n✓ All checkpoint assertions passed")


---

## Reflection questions

1. Why is `df.shape[0]` the number of **rows** and not columns?
2. Which columns would you one-hot encode before linear regression?
3. What additional plots would you create before modeling? *(Lab 4 — Seaborn)*

**Previous:** [Lab 2 — NumPy arrays](lab02_numpy_arrays.ipynb)  
**Next:** [Lab 4 — Seaborn plots](lab04_seaborn_plots.ipynb)
